# AWS Secrets Manager Secrets Import

Secrets Manager names are unique in the account and region. This notebook imports eleven tagged secrets into `aws-import/<account-id>/<region>/` and syncs each one back onto the original name. `secret-key` granularity writes the raw `value` string as a new version. The previous version remains `AWSPREVIOUS`, and the original tags stay on the secret. Vault adds its `hashicorp:vault` tag. Secrets Sync runs in the Vault pod and assumes `mapfre-aws-import-sync`, which can write only `demo-*` secrets. `demo-db-credentials` is a JSON object, which is the only form the AWS console can show on the key/value tab.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=".env")

AWS_REGION = os.getenv("AWS_REGION", "eu-central-1")
os.environ["AWS_REGION"] = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION
os.environ["WORKDIR"] = "/tmp/vault"
print(f"AWS region: {AWS_REGION}")


AWS region: eu-central-1


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

ENV_FILE = next((directory / ".env" for directory in (Path.cwd(), *Path.cwd().parents) if (directory / ".env").is_file()), None)
if ENV_FILE is None:
    raise FileNotFoundError("Could not find the persistent .env file")
load_dotenv(ENV_FILE)

VAULT_TOKEN = os.getenv("VAULT_TOKEN")
VAULT_ADDR = os.getenv("VAULT_ADDR")


In [3]:
# Hashi only
!doormat login -f

import os
import subprocess

# Import the credentials produced by doormat into the notebook kernel.
result = subprocess.run(
    ["bash", "-lc", 'eval "$(doormat aws -a aws_jose.merchan_test export)" && env -0'],
    check=True,
    capture_output=True,
)
for entry in result.stdout.split(b"\0"):
    if entry.startswith(b"AWS_") and b"=" in entry:
        key, value = entry.split(b"=", 1)
        os.environ[key.decode()] = value.decode()

WARN[0001] new doormat cli recommended: v4.6.4 run `doormat update` to update! 
INFO[0001] logging into doormat...                      
INFO[0004] successfully logged into doormat!            


## Authenticate and confirm the AWS account

In [4]:
%%bash
set -euo pipefail
aws sts get-caller-identity \
  --query '{account:Account,arn:Arn}' \
  --output table
printf 'Region\t%s\n' "$AWS_REGION"


------------------------------------------------------------------------------------------------------------------
|                                                GetCallerIdentity                                               |
+---------+------------------------------------------------------------------------------------------------------+
|  account|  492487827579                                                                                        |
|  arn    |  arn:aws:sts::492487827579:assumed-role/aws_jose.merchan_test-developer/jose.merchan@hashicorp.com   |
+---------+------------------------------------------------------------------------------------------------------+
Region	eu-central-1


## Create eleven tagged source secrets

Values and PEM material are generated locally and are never printed by the notebook. Names stay flat, such as `demo-db-password`, because that full name is the Secrets Manager identity in this region.


In [5]:
%%bash
set -euo pipefail
umask 077

DB_PASSWORD="Db-$(openssl rand -hex 16)"
PAYMENTS_API_KEY="pay_$(openssl rand -hex 24)"
MAPS_API_KEY="maps_$(openssl rand -hex 24)"
OAUTH_SECRET="oauth_$(openssl rand -base64 32 | tr -d '\n')"
WEBHOOK_SECRET="whsec_$(openssl rand -hex 24)"
JWT_PRIVATE_KEY=$(openssl genpkey -algorithm EC -pkeyopt ec_paramgen_curve:P-256 2>/dev/null)
SSH_PRIVATE_KEY=$(openssl genpkey -algorithm RSA -pkeyopt rsa_keygen_bits:2048 2>/dev/null)
TLS_CERTIFICATE=$(openssl req -x509 -newkey rsa:2048 -nodes -days 30 \
  -subj '/CN=secrets-import-demo.local' -keyout /dev/null -out /dev/stdout 2>/dev/null)

set_secret() {
  local name=$1 value=$2 category=$3 format=$4
  local deleted
  deleted=$(aws secretsmanager describe-secret --secret-id "$name" --region "$AWS_REGION" \
    --query 'DeletedDate' --output text 2>/dev/null || true)
  if [[ -n "$deleted" && "$deleted" != "None" ]]; then
    aws secretsmanager restore-secret --secret-id "$name" --region "$AWS_REGION" >/dev/null
  fi
  if aws secretsmanager describe-secret --secret-id "$name" --region "$AWS_REGION" >/dev/null 2>&1; then
    aws secretsmanager put-secret-value --secret-id "$name" --secret-string "$value" \
      --region "$AWS_REGION" >/dev/null
    aws secretsmanager tag-resource --secret-id "$name" --region "$AWS_REGION" --tags \
      Key=importable,Value=true Key=migration,Value=aws-secrets-import Key=owner,Value=mapfre \
      Key=environment,Value=demo Key=category,Value="$category" Key=format,Value="$format" >/dev/null
  else
    aws secretsmanager create-secret --name "$name" --secret-string "$value" --region "$AWS_REGION" \
      --tags Key=importable,Value=true Key=migration,Value=aws-secrets-import Key=owner,Value=mapfre \
        Key=environment,Value=demo Key=category,Value="$category" Key=format,Value="$format" >/dev/null
  fi
}

set_secret demo-db-username 'app_import_user' database username
set_secret demo-db-password "$DB_PASSWORD" database password
set_secret demo-payments-api-key "$PAYMENTS_API_KEY" api api-key
set_secret demo-maps-api-key "$MAPS_API_KEY" api api-key
set_secret demo-oauth-client-secret "$OAUTH_SECRET" oauth client-secret
set_secret demo-webhook-signing-secret "$WEBHOOK_SECRET" webhook signing-secret
set_secret demo-jwt-private-key "$JWT_PRIVATE_KEY" crypto pem
set_secret demo-ssh-private-key "$SSH_PRIVATE_KEY" ssh pem
set_secret demo-tls-certificate "$TLS_CERTIFICATE" tls certificate-pem
set_secret demo-connection-string "Server=db.demo.internal;User=app_import_user;Password=${DB_PASSWORD}" database connection-string
DB_CREDENTIALS=$(jq -n --arg username app_import_user --arg password "$DB_PASSWORD" '{username:$username,password:$password}')
set_secret demo-db-credentials "$DB_CREDENTIALS" database json

unset DB_PASSWORD PAYMENTS_API_KEY MAPS_API_KEY OAUTH_SECRET WEBHOOK_SECRET DB_CREDENTIALS
unset JWT_PRIVATE_KEY SSH_PRIVATE_KEY TLS_CERTIFICATE

aws secretsmanager list-secrets --region "$AWS_REGION" --output json | jq -r '
  ["NAME", "CATEGORY", "FORMAT", "IMPORTABLE"],
  (.SecretList[]
    | select(any(.Tags[]?; .Key == "migration" and .Value == "aws-secrets-import"))
    | [ .Name,
        ((.Tags[]? | select(.Key == "category") | .Value) // ""),
        ((.Tags[]? | select(.Key == "format") | .Value) // ""),
        ((.Tags[]? | select(.Key == "importable") | .Value) // "") ])
  | @tsv
'


NAME	CATEGORY	FORMAT	IMPORTABLE
demo-db-username	database	username	true
demo-db-password	database	password	true
demo-payments-api-key	api	api-key	true
demo-maps-api-key	api	api-key	true
demo-oauth-client-secret	oauth	client-secret	true
demo-webhook-signing-secret	webhook	signing-secret	true
demo-jwt-private-key	crypto	pem	true
demo-ssh-private-key	ssh	pem	true
demo-tls-certificate	tls	certificate-pem	true
demo-connection-string	database	connection-string	true
demo-db-credentials	database	json	true


## Activate Secret Import

In [6]:
! vault write -f sys/activation-flags/secrets-import/activate


Key            Value
---            -----
activated      [secrets-import secrets-sync]
unactivated    [enable-scim force-identity-deduplication]


## Write the import plan

Import uses the AWS credentials already present in this shell. No IAM user is created.


In [7]:
%%bash
set -euo pipefail
mkdir -p "$WORKDIR"
umask 077

: "${AWS_ACCESS_KEY_ID:?AWS credentials are required}"
: "${AWS_SECRET_ACCESS_KEY:?AWS credentials are required}"
CALLER_ARN=$(aws sts get-caller-identity --query Arn --output text)

cat > "$WORKDIR/aws-import.hcl" <<EOF
source_aws {
  name = "aws-secrets-manager"
}

destination_vault {
  name    = "vault-kv"
  address = "${VAULT_ADDR}"
  mount   = "aws-import"
}

mapping {
  name        = "tagged-aws-secrets"
  source      = "aws-secrets-manager"
  destination = "vault-kv"
  filter      = "Secret.Tags.importable == \"true\" and Secret.Tags.migration == \"aws-secrets-import\""
}
EOF

echo "Import plan written to $WORKDIR/aws-import.hcl"
echo "AWS caller: $CALLER_ARN"


Import plan written to /tmp/vault/aws-import.hcl
AWS caller: arn:aws:sts::492487827579:assumed-role/aws_jose.merchan_test-developer/jose.merchan@hashicorp.com


## Plan and apply Secrets Import

Only secrets tagged with `importable=true` and `migration=aws-secrets-import` are selected. The import uses the AWS credentials already present in this shell, including the session token.


In [8]:
%%bash
set -euo pipefail

: "${VAULT_ADDR:?VAULT_ADDR is required}"
: "${VAULT_TOKEN:?VAULT_TOKEN is required}"
vault version | grep -q -- '+ent' || {
  echo 'Vault Enterprise CLI is required for operator import.' >&2
  exit 1
}

vault operator import -config="$WORKDIR/aws-import.hcl" plan
vault operator import -config="$WORKDIR/aws-import.hcl" -auto-create -auto-approve apply


-----------
Import plan
-----------
Secrets to be imported to the destination "vault-kv":
  * {aws-secrets-manager demo-db-username}
  * {aws-secrets-manager demo-jwt-private-key}
  * {aws-secrets-manager demo-db-credentials}
  * {aws-secrets-manager demo-payments-api-key}
  * {aws-secrets-manager demo-tls-certificate}
  * {aws-secrets-manager demo-webhook-signing-secret}
  * {aws-secrets-manager demo-oauth-client-secret}
  * {aws-secrets-manager demo-maps-api-key}
  * {aws-secrets-manager demo-ssh-private-key}
  * {aws-secrets-manager demo-connection-string}
  * {aws-secrets-manager demo-db-password}

Secrets that will not be imported from the source "aws-secrets-manager":
  * vault/mapfre-wif-kv/test
  * vault/sync-aws-irsa/verification

-----------
Import plan
-----------
Secrets to be imported to the destination "vault-kv":
  * {aws-secrets-manager demo-ssh-private-key}
  * {aws-secrets-manager demo-oauth-client-secret}
  * {aws-secrets-manager demo-webhook-signing-secret}
  * {aws

## Import into a KV directory

The destination mount stays `aws-import`. A regexp transform writes each secret under `<account-id>/<region>/`, which KV v2 shows as directories. The secret base name stays the Secrets Manager name.


In [9]:
%%bash
set -euo pipefail

mkdir -p "$WORKDIR"
umask 077

AWS_ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)

cat > "$WORKDIR/aws-import-prefixed.hcl" <<EOF
source_aws {
  name = "aws-secrets-manager"
}

destination_vault {
  name    = "vault-kv"
  address = "${VAULT_ADDR}"
  mount   = "aws-import"
}

mapping {
  name        = "tagged-aws-secrets-prefixed"
  source      = "aws-secrets-manager"
  destination = "vault-kv"
  filter      = "Secret.Tags.importable == \"true\" and Secret.Tags.migration == \"aws-secrets-import\""

  transform "regexp" {
    from = "(.+)"
    to   = "${AWS_ACCOUNT_ID}/${AWS_REGION}/\$1"
  }
}
EOF

echo "Import plan written to $WORKDIR/aws-import-prefixed.hcl"


Import plan written to /tmp/vault/aws-import-prefixed.hcl


In [10]:
%%bash
set -euo pipefail

: "${VAULT_ADDR:?VAULT_ADDR is required}"
: "${VAULT_TOKEN:?VAULT_TOKEN is required}"
vault version | grep -q -- '+ent' || {
  echo 'Vault Enterprise CLI is required for operator import.' >&2
  exit 1
}

vault operator import -config="$WORKDIR/aws-import-prefixed.hcl" plan
vault operator import -config="$WORKDIR/aws-import-prefixed.hcl" -auto-create -auto-approve apply

AWS_ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)
echo
echo "Imported names under aws-import/${AWS_ACCOUNT_ID}/${AWS_REGION}/:"
vault kv list "aws-import/${AWS_ACCOUNT_ID}/${AWS_REGION}"


-----------
Import plan
-----------
Secrets to be imported to the destination "vault-kv":
  * {aws-secrets-manager demo-maps-api-key} -> 492487827579/eu-central-1/demo-maps-api-key
  * {aws-secrets-manager demo-oauth-client-secret} -> 492487827579/eu-central-1/demo-oauth-client-secret
  * {aws-secrets-manager demo-ssh-private-key} -> 492487827579/eu-central-1/demo-ssh-private-key
  * {aws-secrets-manager demo-webhook-signing-secret} -> 492487827579/eu-central-1/demo-webhook-signing-secret
  * {aws-secrets-manager demo-db-password} -> 492487827579/eu-central-1/demo-db-password
  * {aws-secrets-manager demo-jwt-private-key} -> 492487827579/eu-central-1/demo-jwt-private-key
  * {aws-secrets-manager demo-db-credentials} -> 492487827579/eu-central-1/demo-db-credentials
  * {aws-secrets-manager demo-db-username} -> 492487827579/eu-central-1/demo-db-username
  * {aws-secrets-manager demo-payments-api-key} -> 492487827579/eu-central-1/demo-payments-api-key
  * {aws-secrets-manager demo-connect

## Verify names without exposing secret values

In [11]:
%%bash
set -euo pipefail

EXPECTED_NAMES=$(aws secretsmanager list-secrets --region "$AWS_REGION" --output json | jq -r '
  .SecretList[]
  | select(any(.Tags[]?; .Key == "migration" and .Value == "aws-secrets-import"))
  | .Name' | sort)
AWS_ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)
IMPORTED_NAMES=$(vault kv list -format=json "aws-import/${AWS_ACCOUNT_ID}/${AWS_REGION}" | jq -r '.[] | rtrimstr("/")' | sort)
diff <(printf '%s\n' "$EXPECTED_NAMES") <(printf '%s\n' "$IMPORTED_NAMES")
COUNT=$(printf '%s\n' "$IMPORTED_NAMES" | grep -c .)
[[ "$COUNT" == 11 ]]
printf 'Imported %s secrets into aws-import/%s/%s/:\n%s\n' "$COUNT" "$AWS_ACCOUNT_ID" "$AWS_REGION" "$IMPORTED_NAMES"


Imported 11 secrets into aws-import/492487827579/eu-central-1/:
demo-connection-string
demo-db-credentials
demo-db-password
demo-db-username
demo-jwt-private-key
demo-maps-api-key
demo-oauth-client-secret
demo-payments-api-key
demo-ssh-private-key
demo-tls-certificate
demo-webhook-signing-secret


## Verify AWS tags against Vault custom metadata

The utility below verifies that every AWS tag exists with the same value in Vault KV v2 `custom_metadata`. Vault-specific metadata such as `import-source` and `operation` is allowed.


In [12]:
%%bash
set -euo pipefail

AWS_ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)

verify_secret_tags_match_vault_metadata() {
  local secret_name=$1
  local aws_tags vault_metadata mismatches

  aws_tags=$(aws secretsmanager describe-secret --secret-id "$secret_name" --region "$AWS_REGION" \
    --query 'Tags' --output json | jq 'map({(.Key): .Value}) | add // {}')
  vault_metadata=$(vault kv metadata get -format=json \
    "aws-import/${AWS_ACCOUNT_ID}/${AWS_REGION}/$secret_name" | jq '.data.custom_metadata // {}')

  mismatches=$(jq -n \
    --argjson aws "$aws_tags" \
    --argjson vault "$vault_metadata" \
    '$aws | to_entries | map(select($vault[.key] != .value))')

  if [[ $(jq 'length' <<<"$mismatches") -ne 0 ]]; then
    echo "ERROR: tag mismatch for $secret_name" >&2
    jq -r '.[] | "  \(.key): AWS=\(.value), Vault=missing-or-different"' \
      <<<"$mismatches" >&2
    return 1
  fi
  printf 'OK %s\n' "$secret_name"
}

VERIFIED_TAGS=0
while IFS= read -r secret_name; do
  [[ -z "$secret_name" ]] && continue
  verify_secret_tags_match_vault_metadata "$secret_name"
  VERIFIED_TAGS=$((VERIFIED_TAGS + 1))
done < <(aws secretsmanager list-secrets --region "$AWS_REGION" --output json | jq -r '
  .SecretList[]
  | select(any(.Tags[]?; .Key == "migration" and .Value == "aws-secrets-import"))
  | select(any(.Tags[]?; .Key == "importable" and .Value == "true"))
  | .Name' | sort)

[[ "$VERIFIED_TAGS" -eq 11 ]]
echo "Verified AWS tags against Vault custom metadata for $VERIFIED_TAGS secrets."


OK demo-connection-string
OK demo-db-credentials
OK demo-db-password
OK demo-db-username
OK demo-jwt-private-key
OK demo-maps-api-key
OK demo-oauth-client-secret
OK demo-payments-api-key
OK demo-ssh-private-key
OK demo-tls-certificate
OK demo-webhook-signing-secret
Verified AWS tags against Vault custom metadata for 11 secrets.


## Sync the imported secrets back to AWS

Secrets Sync runs in the Vault pod. The next cell creates `mapfre-aws-import-sync` and allows `vault-kms-auto-unseal` to assume it. The trust and the pod policy both allow `sts:AssumeRole` and `sts:TagSession`, because EKS Pod Identity session tags are forwarded on role chaining. The sync role can manage only `demo-*` secrets in this region. The pod role does not receive Secrets Manager permissions.

The cell after that writes each imported secret onto the original Secrets Manager name, such as `demo-db-password`, with `role_arn` set on the destination. `secret-key` granularity stores the raw `value` string as a new version. The previous version remains available as `AWSPREVIOUS`. The template emits `.SecretBaseName` only. `.SecretKey` is referenced so Vault accepts the template, and it is not part of the AWS name.

Send `role_arn` on every destination write. A partial write replaces the destination configuration, and Vault would call Secrets Manager as the pod role again.

Removing an association deletes that Secrets Manager secret. These cells do not remove associations, because the template already targets the original names.

In [13]:
%%bash
set -euo pipefail

SYNC_ROLE_NAME=mapfre-aws-import-sync
POD_ROLE_NAME=vault-kms-auto-unseal
POD_ASSUME_POLICY_NAME=mapfre-aws-import-assume-role
SYNC_SECRET_POLICY_NAME=mapfre-aws-import-secrets

AWS_ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)
POD_ROLE_ARN="arn:aws:iam::${AWS_ACCOUNT_ID}:role/${POD_ROLE_NAME}"

TRUST=$(jq -n --arg pod "$POD_ROLE_ARN" '{
  Version: "2012-10-17",
  Statement: [{
    Sid: "TrustVaultPod",
    Effect: "Allow",
    Principal: {AWS: $pod},
    Action: ["sts:AssumeRole", "sts:TagSession"]
  }]
}')

created=false
if aws iam get-role --role-name "$SYNC_ROLE_NAME" >/dev/null 2>&1; then
  aws iam update-assume-role-policy --role-name "$SYNC_ROLE_NAME" --policy-document "$TRUST"
else
  aws iam create-role --role-name "$SYNC_ROLE_NAME" \
    --description "Assumed by Vault Secrets Sync to write imported demo secrets." \
    --assume-role-policy-document "$TRUST" >/dev/null
  created=true
fi

SECRET_POLICY=$(jq -n --arg region "$AWS_REGION" --arg account "$AWS_ACCOUNT_ID" '{
  Version: "2012-10-17",
  Statement: [{
    Sid: "ManageImportedDemoSecrets",
    Effect: "Allow",
    Action: [
      "secretsmanager:CreateSecret",
      "secretsmanager:DeleteSecret",
      "secretsmanager:DescribeSecret",
      "secretsmanager:GetSecretValue",
      "secretsmanager:PutSecretValue",
      "secretsmanager:TagResource",
      "secretsmanager:UntagResource",
      "secretsmanager:UpdateSecret"
    ],
    Resource: "arn:aws:secretsmanager:\($region):\($account):secret:demo-*"
  }]
}')

aws iam put-role-policy --role-name "$SYNC_ROLE_NAME" \
  --policy-name "$SYNC_SECRET_POLICY_NAME" \
  --policy-document "$SECRET_POLICY"

SYNC_ROLE_ARN=$(aws iam get-role --role-name "$SYNC_ROLE_NAME" --query Role.Arn --output text)

ASSUME_POLICY=$(jq -n --arg role "$SYNC_ROLE_ARN" '{
  Version: "2012-10-17",
  Statement: [{
    Sid: "AssumeImportSyncRole",
    Effect: "Allow",
    Action: ["sts:AssumeRole", "sts:TagSession"],
    Resource: $role
  }]
}')

aws iam put-role-policy --role-name "$POD_ROLE_NAME" \
  --policy-name "$POD_ASSUME_POLICY_NAME" \
  --policy-document "$ASSUME_POLICY"

echo "Sync role: $SYNC_ROLE_ARN"
if [[ "$created" == true ]]; then
  echo "Waiting 30s for IAM propagation."
  sleep 30
fi

Sync role: arn:aws:iam::492487827579:role/mapfre-aws-import-sync
Waiting 30s for IAM propagation.


In [14]:
%%bash
set -euo pipefail

SYNC_DESTINATION=aws-import-roundtrip
SYNC_ROLE_NAME=mapfre-aws-import-sync
SYNC_ROLE_ARN=$(aws iam get-role --role-name "$SYNC_ROLE_NAME" --query Role.Arn --output text)

vault write -f sys/activation-flags/secrets-sync/activate >/dev/null || true
vault write "sys/sync/destinations/aws-sm/$SYNC_DESTINATION" \
  region="$AWS_REGION" \
  role_arn="$SYNC_ROLE_ARN" \
  granularity=secret-key \
  secret_name_template='{{ $unused := .SecretKey }}{{ .SecretBaseName }}'

Key                   Value
---                   -----
connection_details    map[region:eu-central-1 role_arn:arn:aws:iam::492487827579:role/mapfre-aws-import-sync]
name                  aws-import-roundtrip
options               map[custom_tags:map[] granularity_level:secret-key secret_name_template:{{ $unused := .SecretKey }}{{ .SecretBaseName }}]
type                  aws-sm


In [15]:
%%bash
set -euo pipefail

SYNC_DESTINATION=aws-import-roundtrip
AWS_ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)
while IFS= read -r base_name; do
  [[ -z "$base_name" || "$base_name" == */ ]] && continue
  secret_name="${AWS_ACCOUNT_ID}/${AWS_REGION}/${base_name%/}"
  DATA_KEYS=$(vault kv get -format=json "aws-import/$secret_name" | \
    jq -r '.data.data | keys | join(",")')
  if [[ "$DATA_KEYS" != "value" ]]; then
    echo "ERROR: $secret_name does not contain exactly the imported key 'value'." >&2
    exit 1
  fi
  vault write \
    "sys/sync/destinations/aws-sm/$SYNC_DESTINATION/associations/set" \
    mount=aws-import \
    secret_name="$secret_name" >/dev/null
  echo "Associated $secret_name"
done < <(vault kv list -format=json "aws-import/${AWS_ACCOUNT_ID}/${AWS_REGION}" | jq -r '.[]' | sort)


Associated 492487827579/eu-central-1/demo-connection-string
Associated 492487827579/eu-central-1/demo-db-credentials
Associated 492487827579/eu-central-1/demo-db-password
Associated 492487827579/eu-central-1/demo-db-username
Associated 492487827579/eu-central-1/demo-jwt-private-key
Associated 492487827579/eu-central-1/demo-maps-api-key
Associated 492487827579/eu-central-1/demo-oauth-client-secret
Associated 492487827579/eu-central-1/demo-payments-api-key
Associated 492487827579/eu-central-1/demo-ssh-private-key
Associated 492487827579/eu-central-1/demo-tls-certificate
Associated 492487827579/eu-central-1/demo-webhook-signing-secret


## Verify round-trip names, values, and sync status

The check compares SHA-256 digests of the imported `value` key with the original Secrets Manager secret of the same name. It does not print secret values.


In [16]:
%%bash
set -euo pipefail

SYNC_DESTINATION=aws-import-roundtrip
AWS_ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)
VAULT_NAMES=$(vault kv list -format=json "aws-import/${AWS_ACCOUNT_ID}/${AWS_REGION}" | jq -r '.[]' | sort)

for attempt in {1..12}; do
  ASSOCIATIONS=$(vault read -format=json \
    "sys/sync/destinations/aws-sm/$SYNC_DESTINATION/associations")
  SYNCED_COUNT=$(jq '[.data.associated_secrets[] | select(.sync_status == "SYNCED")] | length' \
    <<<"$ASSOCIATIONS")
  [[ "$SYNCED_COUNT" -eq 11 ]] && break
  sleep 5
done

VERIFIED_VALUES=0
while IFS= read -r base_name; do
  [[ -z "$base_name" || "$base_name" == */ ]] && continue
  secret_name="${AWS_ACCOUNT_ID}/${AWS_REGION}/${base_name%/}"
  aws_name="${base_name%/}"
  VAULT_VALUE=$(vault kv get -field=value "aws-import/$secret_name")
  AWS_VALUE=$(aws secretsmanager get-secret-value --secret-id "$aws_name" \
    --region "$AWS_REGION" --query SecretString --output text)
  VAULT_DIGEST=$(printf '%s' "$VAULT_VALUE" | \
    openssl dgst -sha256 -r | awk '{print $1}')
  AWS_DIGEST=$(printf '%s' "$AWS_VALUE" | \
    openssl dgst -sha256 -r | awk '{print $1}')
  unset VAULT_VALUE AWS_VALUE
  [[ "$VAULT_DIGEST" == "$AWS_DIGEST" ]] || {
    echo "ERROR: value mismatch for $secret_name" >&2
    exit 1
  }
  VERIFIED_VALUES=$((VERIFIED_VALUES + 1))
  echo "OK $secret_name"
done <<<"$VAULT_NAMES"

TOTAL_COUNT=$(jq '.data.associated_secrets | length' <<<"$ASSOCIATIONS")
[[ "$VERIFIED_VALUES" -eq 11 && "$SYNCED_COUNT" -eq 11 && "$TOTAL_COUNT" -eq 11 ]]
echo "Verified 11 values and SYNCED associations in AWS Secrets Manager."

OK 492487827579/eu-central-1/demo-connection-string
OK 492487827579/eu-central-1/demo-db-credentials
OK 492487827579/eu-central-1/demo-db-password
OK 492487827579/eu-central-1/demo-db-username
OK 492487827579/eu-central-1/demo-jwt-private-key
OK 492487827579/eu-central-1/demo-maps-api-key
OK 492487827579/eu-central-1/demo-oauth-client-secret
OK 492487827579/eu-central-1/demo-payments-api-key
OK 492487827579/eu-central-1/demo-ssh-private-key
OK 492487827579/eu-central-1/demo-tls-certificate
OK 492487827579/eu-central-1/demo-webhook-signing-secret
Verified 11 values and SYNCED associations in AWS Secrets Manager.


## Optional cleanup

Set `RUN_AWS_IMPORT_CLEANUP=true` to delete the Secrets Manager secrets, the `aws-import/` mount, the sync destination, the `mapfre-aws-import-sync` role, and the pod policy that allows assuming it. Deletion skips the recovery window so the same names can be created again.

In [ ]:
%%bash
set -euo pipefail
RUN_AWS_IMPORT_CLEANUP=${RUN_AWS_IMPORT_CLEANUP:-false}
if [[ "$RUN_AWS_IMPORT_CLEANUP" != "true" ]]; then
  echo "Cleanup skipped. Set RUN_AWS_IMPORT_CLEANUP=true to execute it."
  exit 0
fi

SYNC_DESTINATION_PATH="sys/sync/destinations/aws-sm/aws-import-roundtrip"
if vault read "$SYNC_DESTINATION_PATH" >/dev/null 2>&1; then
  vault delete "$SYNC_DESTINATION_PATH" purge=true
  for attempt in {1..24}; do
    if ! vault read "$SYNC_DESTINATION_PATH" >/dev/null 2>&1; then
      break
    fi
    if [[ "$attempt" -eq 24 ]]; then
      echo 'ERROR: Secrets Sync destination was not purged.' >&2
      exit 1
    fi
    sleep 5
  done
fi

while IFS= read -r secret_name; do
  [[ -z "$secret_name" ]] && continue
  aws secretsmanager delete-secret --secret-id "$secret_name" --region "$AWS_REGION" \
    --force-delete-without-recovery >/dev/null || true
done < <(aws secretsmanager list-secrets --region "$AWS_REGION" --output json | jq -r '
  .SecretList[]
  | select(any(.Tags[]?; .Key == "migration" and .Value == "aws-secrets-import"))
  | .Name')

if vault secrets list -format=json | jq -e 'has("aws-import/")' >/dev/null; then
  vault secrets disable aws-import/
fi

rm -f "$WORKDIR/aws-import.hcl" "$WORKDIR/aws-import-prefixed.hcl"

SYNC_ROLE_NAME=mapfre-aws-import-sync
POD_ROLE_NAME=vault-kms-auto-unseal
aws iam delete-role-policy --role-name "$POD_ROLE_NAME" \
  --policy-name mapfre-aws-import-assume-role || true
aws iam delete-role-policy --role-name "$SYNC_ROLE_NAME" \
  --policy-name mapfre-aws-import-secrets || true
aws iam delete-role --role-name "$SYNC_ROLE_NAME" || true

echo "AWS import resources deleted and aws-import/ disabled."